In [0]:
endpoint = "frentes_detalhes"

table_name = endpoint

print(f"Endpoint: {endpoint}")
print(f"Tabela: {table_name}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

from datetime import datetime

In [0]:
%run ../utils/logger

In [0]:
%run ../utils/metadata_manager

In [0]:
%run ../utils/delta_writer

In [0]:
# ==========================================
# CAMINHOS
# ==========================================

bronze_path = (
    f"/Volumes/workspace/default/"
    f"camara_deputados/bronze/{endpoint}"
)

silver_table = (
    f"workspace.silver.{table_name}"
)

print(f"Bronze path: {bronze_path}")
print(f"Silver table: {silver_table}")

In [0]:
# ==========================================
# LEITURA BRONZE
# ==========================================

df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

display(df)

In [0]:
# ==========================================
# TRANSFORMAÇÕES SILVER
# ==========================================

# ------------------------------------------
# CONVERSÃO COLUNAS COMPLEXAS
# ------------------------------------------

for field in df.schema.fields:

    field_type = str(field.dataType)

    if (
        "MapType" in field_type
        or "ArrayType" in field_type
        or "StructType" in field_type
    ):

        df = df.withColumn(
            field.name,
            F.to_json(
                F.col(field.name)
            )
        )

# ------------------------------------------
# TIMESTAMP PROCESSAMENTO
# ------------------------------------------

df = (
    df
    .withColumn(
        "dt_processamento",
        F.current_timestamp()
    )
)

# ------------------------------------------
# REMOVE DUPLICADOS
# ------------------------------------------

df = df.dropDuplicates()

In [0]:
# ==========================================
# VALIDAÇÃO DATAFRAME VAZIO
# ==========================================

record_count = df.count()

print(
    f"Registros processados: {record_count}"
)

if record_count == 0:

    log_info(
        f"Nenhum dado para processar: {endpoint}"
    )

    dbutils.notebook.exit(
        "Sem dados"
    )


In [0]:
# ==========================================
# ESCRITA SILVER
# ==========================================

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(
    f"Tabela Silver salva: {silver_table}"
)


In [0]:
# ==========================================
# REGISTRO METADATA
# ==========================================

register_execution(
    table_name=f"silver.{table_name}",
    endpoint=endpoint,
    status="SUCCESS",
    record_count=record_count,
    error_message=None
)

In [0]:
# ==========================================
# FINALIZAÇÃO
# ==========================================

log_info(
    f"Processamento Silver finalizado: {endpoint}"
)